# 스킬 & 플러그인 시스템 — Skills and Plugins

**Introduction to Claude Cowork — L04~L06 대응**

이 노트북에서 다루는 내용:
1. 마크다운 스킬 파일 작성 → system prompt로 주입하여 전문가 행동 유도
2. Instructions 패턴으로 프로젝트 컨텍스트 설정
3. 예약 작업 개념을 코드로 시뮬레이션

> **핵심 원리**: Cowork의 스킬은 마크다운 파일이 system prompt에 주입되는 것과 동일합니다. API에서도 같은 패턴을 적용할 수 있습니다.

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import json
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 스킬 = System Prompt

Cowork의 스킬은 마크다운 파일입니다. Claude가 이 파일을 읽고 해당 분야의 전문가처럼 행동합니다.
API에서는 이것이 **system prompt에 마크다운을 주입**하는 것과 동일합니다.

### 스킬 없이 vs 스킬 있이

In [ ]:
# ── 스킬 없이: 일반적인 보고서 요청 ──────────────────
response_no_skill = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": "프로젝트 현황 보고서를 작성해줘. 진행률 75%, 예산 80% 소진, 일정 1주 지연."
    }]
)
print("=== 스킬 없이 ===")
print(response_no_skill.content[0].text[:500])

In [ ]:
# ── 스킬 정의: 마크다운 형식의 보고서 작성 스킬 ──────
WEEKLY_REPORT_SKILL = """
# /weekly-report 스킬

## 목적
주간 프로젝트 현황 보고서를 표준 형식으로 작성합니다.

## 출력 형식
반드시 다음 구조를 따르세요:

### 1. 요약 (Executive Summary)
- 한 문장으로 전체 상황 요약
- RAG 상태 표시: 🟢 정상 / 🟡 주의 / 🔴 위험

### 2. 핵심 지표
| 지표 | 현재 | 목표 | 상태 |
|------|------|------|------|
| 진행률 | X% | Y% | 🟢/🟡/🔴 |

### 3. 이슈 & 리스크
- 각 이슈에 대해: [심각도] 내용 → 대응 방안

### 4. 다음 주 계획
- 우선순위가 높은 항목부터 3개

## 규칙
- 수치에는 항상 단위를 표기하세요
- 정량적 판단: 진행률 >90% = 🟢, 70-90% = 🟡, <70% = 🔴
- 예산 소진율이 진행률보다 5%p 이상 높으면 🔴
"""

# 스킬을 system prompt로 주입
response_with_skill = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=WEEKLY_REPORT_SKILL,
    messages=[{
        "role": "user",
        "content": "프로젝트 현황 보고서를 작성해줘. 진행률 75%, 예산 80% 소진, 일정 1주 지연."
    }]
)
print("=== 스킬 적용 후 ===")
print(response_with_skill.content[0].text)

## §2. Instructions = 프로젝트 컨텍스트

Cowork 프로젝트의 **Instructions**는 모든 세션에 자동으로 적용되는 맥락입니다.
API에서는 system prompt에 프로젝트 정보를 포함시키는 것과 동일합니다.

In [ ]:
# ── Instructions: 프로젝트 맥락 정의 ──────────────────
PROJECT_INSTRUCTIONS = """
# 프로젝트 Instructions

## 프로젝트 정보
- 프로젝트명: 경희대학교 신공학관 구조설계
- PM: 김구조 교수
- 클라이언트: 경희대학교 시설팀 (담당: 이시설 과장)

## 관련 인물
- 김구조: 구조설계 총괄 (kim@khu.ac.kr)
- 이시설: 시설팀 담당자 (lee@khu.ac.kr)
- 박건축: 건축설계 (park@arch.com)
- 최감리: 감리단장 (choi@supervision.com)

## 파일 위치
- 구조계산서: ./structural/
- 도면: ./drawings/
- 회의록: ./meetings/
- 보고서 출력: ./reports/

## 출력 규칙
- 모든 수치는 SI 단위 (kN, mm, MPa)
- KDS 41 10 15 기준 적용
- 보고서는 .docx, 계산서는 .xlsx
"""

# Instructions + 스킬을 조합
combined_system = PROJECT_INSTRUCTIONS + "\n\n" + WEEKLY_REPORT_SKILL

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=combined_system,
    messages=[{
        "role": "user",
        "content": "이번 주 현황 보고서를 작성해줘. 기초공사 진행률 85%, 예산 70% 소진, 이시설 과장에게 보낼 거야."
    }]
)
print("=== Instructions + 스킬 조합 결과 ===")
print(response.content[0].text)

## §3. 플러그인 = 스킬 + 커넥터 번들

실제 Cowork 플러그인은 폴더 구조입니다. 여기서는 여러 스킬을 묶어 "플러그인"처럼 동작시킵니다.

In [ ]:
# ── 플러그인 시뮬레이션: 여러 스킬을 로드하여 번들링 ──
# 실제 Cowork 플러그인 폴더 구조를 코드로 재현

plugin_dir = "ae_plugin"
skills_dir = os.path.join(plugin_dir, "skills")
os.makedirs(skills_dir, exist_ok=True)

# 스킬 1: 주간 보고서
with open(os.path.join(skills_dir, "weekly_report.md"), "w", encoding="utf-8") as f:
    f.write(WEEKLY_REPORT_SKILL)

# 스킬 2: 구조 검토
STRUCTURAL_REVIEW_SKILL = """
# /structural-review 스킬

## 목적
구조부재의 설계 적정성을 검토합니다.

## 검토 항목
1. **휨 강도**: Mu ≤ φMn (φ = 0.85)
2. **전단 강도**: Vu ≤ φVn (φ = 0.75)
3. **처짐**: δmax ≤ L/240 (활하중), L/360 (정밀기기)
4. **균열폭**: wmax ≤ 0.3mm (일반), 0.2mm (수밀)

## 출력 형식
| 부재 | 검토항목 | 설계값 | 허용값 | D/C ratio | 판정 |
|------|----------|--------|--------|-----------|------|

## 규칙
- D/C ratio > 1.0이면 NG 판정
- D/C ratio 0.9~1.0이면 "주의" 표시
- KDS 41 10 15 / KDS 14 20 기준 적용
"""
with open(os.path.join(skills_dir, "structural_review.md"), "w", encoding="utf-8") as f:
    f.write(STRUCTURAL_REVIEW_SKILL)

# 플러그인 매니페스트
plugin_manifest = {
    "name": "ae-structural-plugin",
    "description": "건축공학 구조설계 전문 플러그인",
    "version": "1.0.0",
    "skills": ["weekly_report", "structural_review"],
    "connectors": ["local_files"]
}
with open(os.path.join(plugin_dir, "plugin.json"), "w", encoding="utf-8") as f:
    json.dump(plugin_manifest, f, indent=2, ensure_ascii=False)

print("=== 플러그인 폴더 구조 ===")
for root, dirs, files in os.walk(plugin_dir):
    level = root.replace(plugin_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

In [ ]:
# ── 플러그인에서 스킬 로드 → system prompt 조합 ──────
def load_plugin(plugin_path: str) -> str:
    """플러그인 폴더에서 모든 스킬을 읽어 system prompt를 구성한다."""
    skills_path = os.path.join(plugin_path, "skills")
    combined = ""
    
    # 매니페스트 읽기
    manifest_path = os.path.join(plugin_path, "plugin.json")
    if os.path.exists(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)
        combined += f"# Plugin: {manifest['name']}\n{manifest['description']}\n\n"
    
    # 모든 스킬 파일 로드
    for skill_file in sorted(os.listdir(skills_path)):
        if skill_file.endswith(".md"):
            with open(os.path.join(skills_path, skill_file), "r", encoding="utf-8") as f:
                combined += f.read() + "\n\n"
    
    return combined

# 플러그인 로드
plugin_system = load_plugin("ae_plugin")
print(f"=== 로드된 플러그인 시스템 프롬프트 ({len(plugin_system)}자) ===")
print(plugin_system[:200] + "...")

# 플러그인 + Instructions 조합하여 구조 검토 실행
full_system = PROJECT_INSTRUCTIONS + "\n\n" + plugin_system

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=full_system,
    messages=[{
        "role": "user",
        "content": (
            "B1 보의 구조 검토를 해줘.\n"
            "- 단면: 400x600mm, fck=30MPa, fy=400MPa\n"
            "- 설계 휨모멘트 Mu=320kN·m, 설계 전단력 Vu=180kN\n"
            "- 주근 4-D25 (As=2027mm²), 스터럽 D10@200"
        )
    }]
)
print("\n=== 플러그인 기반 구조 검토 결과 ===")
print(response.content[0].text)

## §4. 핵심 정리

- **스킬 = system prompt**: 마크다운 파일에 워크플로를 기술하면 Claude가 전문가처럼 행동한다.
- **Instructions = 프로젝트 컨텍스트**: 인물, 파일 위치, 출력 규칙 등을 설정하면 약칭이 동작한다.
- **플러그인 = 스킬 번들**: 여러 스킬 + 커넥터 + 매니페스트를 폴더로 묶은 것. 편집 가능한 평문 파일.
- **실제 Cowork에서**: GUI로 플러그인을 설치하고, Instructions 패널에 맥락을 입력하면 동일 효과.
- **다음 노트북**: `CW_03_ae_cowork_design.ipynb`에서 건축공학 도메인 Cowork 프로젝트를 설계합니다.